## Data Availability Notice

The data used in this notebook was provided by **Dogs Trust** under a formal data sharing agreement as part of an MSc Data Science dissertation at Liverpool John Moores University, supervised by Dr Jon Bielby.

**The raw data files are not included in this repository.** To reproduce the analysis, the following data extracts from the Dogs Trust Salesforce CRM are required:

- `Dog Affiliation and Status_Sub Status History Pull.xlsx` (Affiliation History sheet)
- `Sub Status History v2.csv`

Requests for data access should be directed to the Dogs Trust Data Science & Analytics team.

All data handling was conducted in accordance with GDPR and Dogs Trust's data governance protocols.

In [ ]:
import simpy
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats as scipy_stats

# Load affiliation history
affiliation_history = pd.read_excel(
    '../data/[Dogs Trust data file - not included in repo]',
    sheet_name='Affiliation History'
)

print(f"Affiliation History loaded: {affiliation_history.shape}")

In [ ]:
# Simulation parameters (from 02_simulation_inputs.ipynb)
RANDOM_SEED = 42
SIM_DURATION = 365          # simulate 1 year (in days)
ARRIVAL_RATE = 628 / 30     # average dogs per day

# Stage duration distributions
INTAKE_SHAPE = 0.9319
INTAKE_SCALE = 26.4569
MOVEMENT_SHAPE = 1.0914
MOVEMENT_SCALE = 11.2346
TRANSFER_RATE = 1 / 18
RETURN_RATE = 1 / 41.9

# Transition probabilities
P_INTAKE_TO_ADOPTION = 0.420
P_INTAKE_TO_TRANSFER = 0.077

P_MOVEMENT_TO_ADOPTION = 0.383
P_MOVEMENT_TO_MOVEMENT = 0.387
P_MOVEMENT_TO_TRANSFER = 0.167

# CALIBRATED via sensitivity analysis in 03_simulation_model.ipynb
# P=0.30 produces mean LOS of 58.87 days, matching real data mean of 58.9 days
P_ADOPTION_TO_RETURN = 0.30

print("Calibrated parameters defined (P_ADOPTION_TO_RETURN = 0.30)")

In [ ]:
def transfer_stage(env):
    yield env.timeout(np.random.exponential(1 / TRANSFER_RATE))

def return_stage(env):
    yield env.timeout(np.random.exponential(1 / RETURN_RATE))

def dog_journey(env, dog_id, results):
    arrival_time = env.now

    # INTAKE STAGE
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    # Decision after Intake
    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los, 'movement': 0})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    # MOVEMENT STAGE (loop)
    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        rand = np.random.random()
        total = P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER
        if rand < P_MOVEMENT_TO_ADOPTION / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
                movement_count += 1
            break
        elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los, 'movement': movement_count})

def arrival_generator(env, results):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey(env, dog_id, results))

print("Calibrated single-return model defined")

In [ ]:
N_REPLICATIONS = 100
all_results = []

for rep in range(N_REPLICATIONS):
    np.random.seed(rep)
    rep_results = []

    env = simpy.Environment()
    env.process(arrival_generator(env, rep_results))
    env.run(until=SIM_DURATION)

    rep_df = pd.DataFrame(rep_results)
    rep_df['replication'] = rep
    all_results.append(rep_df)

mc_df = pd.concat(all_results, ignore_index=True)

print(f"Calibrated baseline (P_ADOPTION_TO_RETURN = 0.30)")
print(f"Total dogs across all replications: {len(mc_df)}")
print(f"\nMean LOS per replication:")
print(mc_df.groupby('replication')['los'].mean().describe().round(2))

In [ ]:
bins = [0, 60, 182, float('inf')]
labels = ['0-60 days (Normal)', '60-182 days (Medium)', '183+ days (Long-term)']

mc_df['LOS Band'] = pd.cut(mc_df['los'], bins=bins, labels=labels)
sim_band_props = mc_df['LOS Band'].value_counts(normalize=True).sort_index() * 100

real_band_props = pd.Series({
    '0-60 days (Normal)': 71.6,
    '60-182 days (Medium)': 16.7,
    '183+ days (Long-term)': 7.3
})

comparison = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'Calibrated Model (%)': sim_band_props.round(1)
})

# Confidence intervals
rep_means = mc_df.groupby('replication')['los'].mean()
mean_los = rep_means.mean()
ci_lower = mean_los - 1.96 * rep_means.std()
ci_upper = mean_los + 1.96 * rep_means.std()

print("Calibrated Model Validation:")
print(f"Mean LOS: {mean_los:.2f} days (95% CI: {ci_lower:.2f}–{ci_upper:.2f})")
print(f"Real Data Mean LOS: 58.9 days")
print(f"Deviation: {abs(mean_los - 58.9) / 58.9 * 100:.2f}%\n")
print(comparison)

In [ ]:
# RQ2 — Kennel Capacity Sweep (Calibrated Model, P=0.30)

def dog_journey_capacity(env, dog_id, kennels, results):
    arrival_time = env.now
    with kennels.request() as request:
        yield request
        wait_time = env.now - arrival_time

        intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
        yield env.timeout(intake_duration)

        rand = np.random.random()
        if rand < P_INTAKE_TO_ADOPTION:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            los = env.now - arrival_time
            results.append({'dog_id': dog_id, 'los': los, 'wait_time': wait_time})
            return
        elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
            yield from transfer_stage(env)

        movement_count = 0
        while True:
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            movement_count += 1
            rand = np.random.random()
            total = P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER
            if rand < P_MOVEMENT_TO_ADOPTION / total:
                if np.random.random() < P_ADOPTION_TO_RETURN:
                    yield from return_stage(env)
                    yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
                break
            elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / total:
                yield from transfer_stage(env)

        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los, 'wait_time': wait_time})

def arrival_generator_capacity(env, kennels, results):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_capacity(env, dog_id, kennels, results))

capacities = [200, 400, 600, 800, 1000]
N_REPLICATIONS = 100
capacity_results = []

for capacity in capacities:
    rep_los = []
    rep_wait = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        kennels = simpy.Resource(env, capacity=capacity)
        env.process(arrival_generator_capacity(env, kennels, rep_results))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        if len(rep_df) > 0:
            rep_los.append(rep_df['los'].mean())
            rep_wait.append(rep_df['wait_time'].mean())

    capacity_results.append({
        'Capacity': capacity,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Wait Time': round(np.mean(rep_wait), 2),
        'Wait 95% CI Lower': round(np.mean(rep_wait) - 1.96 * np.std(rep_wait), 2),
        'Wait 95% CI Upper': round(np.mean(rep_wait) + 1.96 * np.std(rep_wait), 2),
    })
    print(f"Capacity {capacity} done — Mean LOS: {np.mean(rep_los):.2f}, Mean Wait: {np.mean(rep_wait):.2f}")

capacity_df = pd.DataFrame(capacity_results)
print("\nRQ2 Capacity Sweep Results (Calibrated Model):")
print(capacity_df.to_string(index=False))

In [ ]:
fig_capacity = go.Figure()

fig_capacity.add_trace(go.Scatter(
    x=capacity_df['Capacity'], 
    y=capacity_df['Mean LOS'],
    mode='lines+markers+text', 
    name='Mean LOS (days)',
    text=capacity_df['Mean LOS'].round(1),
    textposition='top center',
    textfont=dict(size=11, color='black'),
    marker=dict(size=8, color='steelblue'),
    line=dict(color='steelblue', width=2),
    error_y=dict(type='data', symmetric=False,
        array=capacity_df['LOS 95% CI Upper'] - capacity_df['Mean LOS'],
        arrayminus=capacity_df['Mean LOS'] - capacity_df['LOS 95% CI Lower'], visible=True)
))

fig_capacity.add_trace(go.Scatter(
    x=capacity_df['Capacity'], 
    y=capacity_df['Mean Wait Time'],
    mode='lines+markers+text', 
    name='Mean Wait Time (days)',
    text=capacity_df['Mean Wait Time'].round(1),
    textposition='bottom center',
    textfont=dict(size=11, color='black'),
    marker=dict(size=8, color='salmon'),
    line=dict(color='salmon', width=2),
    error_y=dict(type='data', symmetric=False,
        array=capacity_df['Wait 95% CI Upper'] - capacity_df['Mean Wait Time'],
        arrayminus=capacity_df['Mean Wait Time'] - capacity_df['Wait 95% CI Lower'], visible=True)
))

fig_capacity.update_layout(
    title='RQ2 — Effect of Kennel Capacity on LOS and Waiting Time (Calibrated Model, 100 Replications, 95% CI)',
    xaxis_title='Kennel Capacity (Number of Spaces)',
    yaxis_title='Duration (Days)',
    height=500, width=950,
    legend=dict(x=0.65, y=1.15)
)


fig_capacity.write_image('../outputs/rq2_capacity_sweep_calibrated.png', width=1100, height=600, scale=2)
fig_capacity.write_html('../outputs/rq2_capacity_sweep_calibrated.html')

fig_capacity.show()

In [ ]:
# RQ3 Scenario 1 — Foster Network Expansion (Calibrated Model, P=0.30)

def dog_journey_foster(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer):
    arrival_time = env.now
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1
        rand = np.random.random()
        if rand < p_move_adopt:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < p_move_adopt + p_move_transfer:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los})

def arrival_gen_foster(env, results, p_move_adopt, p_move_move, p_move_transfer):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_foster(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer))

foster_scenarios = {
    'Baseline (50% foster)': {'p_move_adopt': 0.383, 'p_move_move': 0.387},
    'Expanded (60% foster)': {'p_move_adopt': 0.420, 'p_move_move': 0.350},
    'High Expansion (70% foster)': {'p_move_adopt': 0.460, 'p_move_move': 0.310},
    'Full Expansion (80% foster)': {'p_move_adopt': 0.500, 'p_move_move': 0.270},
}

N_REPLICATIONS = 100
foster_results = []

for scenario_name, params in foster_scenarios.items():
    p_move_adopt = params['p_move_adopt']
    p_move_move = params['p_move_move']
    p_move_transfer = 1 - p_move_adopt - p_move_move

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_foster(env, rep_results, p_move_adopt, p_move_move, p_move_transfer))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    foster_results.append({
        'Scenario': scenario_name,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
    })
    print(f"{scenario_name} done — Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

foster_df = pd.DataFrame(foster_results)
print("\nRQ3 Scenario 1 — Foster Network Expansion (Calibrated):")
print(foster_df.to_string(index=False))

In [ ]:
# RQ3 Scenario 1 - Foster Network Expansion plot
fig_s1 = go.Figure()
fig_s1.add_trace(go.Bar(
    x=foster_df['Scenario'],
    y=foster_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(type='data', symmetric=False,
        array=foster_df['LOS 95% CI Upper'] - foster_df['Mean LOS'],
        arrayminus=foster_df['Mean LOS'] - foster_df['LOS 95% CI Lower'], visible=True)
))
fig_s1.add_trace(go.Scatter(
    x=foster_df['Scenario'],
    y=foster_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers+text',
    text=foster_df['Mean Dogs Rehomed'].astype(int),
    textposition='top center',
    line=dict(color='red', width=2),
    yaxis='y2'
))
fig_s1.update_layout(
    title='RQ3 Scenario 1 — Foster Network Expansion: LOS Reduction and Rehoming Throughput (Calibrated Model)',
    xaxis_title='Foster Proportion',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(title='Dogs Rehomed per Year', overlaying='y', side='right'),
    height=550, width=1000,
    legend=dict(x=0.35, y=1.15)
)
fig_s1.write_image('../outputs/rq3_scenario1_foster_expansion.png', width=1200, height=650, scale=2)
fig_s1.write_html('../outputs/rq3_scenario1_foster_expansion.html')
fig_s1.show()

In [ ]:
# RQ3 Scenario 2 — Longest-Stay Prioritisation (Calibrated Model, P=0.30)

def dog_journey_priority(env, dog_id, results, threshold, boost):
    arrival_time = env.now
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        if movement_count >= threshold:
            p_adopt = min(P_MOVEMENT_TO_ADOPTION + boost, 0.90)
            p_move = P_MOVEMENT_TO_MOVEMENT - boost / 2
            p_transfer = P_MOVEMENT_TO_TRANSFER - boost / 2
        else:
            p_adopt = P_MOVEMENT_TO_ADOPTION
            p_move = P_MOVEMENT_TO_MOVEMENT
            p_transfer = P_MOVEMENT_TO_TRANSFER

        total = p_adopt + p_move + p_transfer
        rand = np.random.random()
        if rand < p_adopt / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (p_adopt + p_transfer) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los})

def arrival_gen_priority(env, results, threshold, boost):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_priority(env, dog_id, results, threshold, boost))

priority_scenarios = {
    'Baseline (no priority)': {'threshold': 999, 'boost': 0.0},
    'Priority after 2 moves': {'threshold': 2, 'boost': 0.15},
    'Priority after 3 moves': {'threshold': 3, 'boost': 0.15},
    'Priority after 2 moves (high boost)': {'threshold': 2, 'boost': 0.25},
}

N_REPLICATIONS = 100
priority_results = []

for scenario_name, params in priority_scenarios.items():
    threshold = params['threshold']
    boost = params['boost']

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_priority(env, rep_results, threshold, boost))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    priority_results.append({
        'Scenario': scenario_name,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
    })
    print(f"{scenario_name} done — Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

priority_df = pd.DataFrame(priority_results)
print("\nRQ3 Scenario 2 — Longest-Stay Prioritisation (Calibrated):")
print(priority_df.to_string(index=False))

In [ ]:
# RQ3 Scenario 2 - Longest-Stay Prioritisation plot
fig_s2 = go.Figure()
fig_s2.add_trace(go.Bar(
    x=priority_df['Scenario'],
    y=priority_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(type='data', symmetric=False,
        array=priority_df['LOS 95% CI Upper'] - priority_df['Mean LOS'],
        arrayminus=priority_df['Mean LOS'] - priority_df['LOS 95% CI Lower'], visible=True)
))
fig_s2.add_trace(go.Scatter(
    x=priority_df['Scenario'],
    y=priority_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers+text',
    text=priority_df['Mean Dogs Rehomed'].astype(int),
    textposition='top center',
    line=dict(color='red', width=2),
    yaxis='y2'
))
fig_s2.update_layout(
    title='RQ3 Scenario 2 — Longest-Stay Prioritisation: LOS Reduction and Rehoming Throughput (Calibrated Model)',
    xaxis_title='Priority Configuration',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(title='Dogs Rehomed per Year', overlaying='y', side='right'),
    height=550, width=1000,
    legend=dict(x=0.35, y=1.15)
)
fig_s2.write_image('../outputs/rq3_scenario2_priority.png', width=1200, height=650, scale=2)
fig_s2.write_html('../outputs/rq3_scenario2_priority.html')
fig_s2.show()

In [ ]:
# RQ3 Scenario 3 — Combined Intervention (Calibrated Model, P=0.30)

def dog_journey_combined(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost):
    arrival_time = env.now
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        if movement_count >= threshold:
            p_adopt = min(p_move_adopt + boost, 0.90)
            p_move = p_move_move - boost / 2
            p_trans = p_move_transfer - boost / 2
        else:
            p_adopt = p_move_adopt
            p_move = p_move_move
            p_trans = p_move_transfer

        total = p_adopt + p_move + p_trans
        rand = np.random.random()
        if rand < p_adopt / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (p_adopt + p_trans) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los})

def arrival_gen_combined(env, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_combined(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost))

combined_scenarios = {
    'Baseline': {'p_move_adopt': 0.383, 'p_move_move': 0.387, 'threshold': 999, 'boost': 0.0},
    'Foster Only (80%)': {'p_move_adopt': 0.500, 'p_move_move': 0.270, 'threshold': 999, 'boost': 0.0},
    'Priority Only (high boost)': {'p_move_adopt': 0.383, 'p_move_move': 0.387, 'threshold': 2, 'boost': 0.25},
    'Combined (Foster + Priority)': {'p_move_adopt': 0.500, 'p_move_move': 0.270, 'threshold': 2, 'boost': 0.25},
}

N_REPLICATIONS = 100
combined_results = []

for scenario_name, params in combined_scenarios.items():
    p_move_adopt = params['p_move_adopt']
    p_move_move = params['p_move_move']
    p_move_transfer = 1 - p_move_adopt - p_move_move
    threshold = params['threshold']
    boost = params['boost']

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_combined(env, rep_results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    baseline_dogs = 6037  # calibrated baseline
    combined_results.append({
        'Scenario': scenario_name,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
        'Extra Dogs Rehomed': round(np.mean(rep_adopted) - baseline_dogs, 0),
        'LOS Reduction': round(58.87 - np.mean(rep_los), 2)
    })
    print(f"{scenario_name} done — Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

combined_df = pd.DataFrame(combined_results)
print("\nRQ3 Scenario 3 — Combined Intervention (Calibrated):")
print(combined_df.to_string(index=False))

In [ ]:
# Consolidated RQ3 headline comparison
fig_headline = go.Figure()

fig_headline.add_trace(go.Bar(
    x=combined_df['Scenario'],
    y=combined_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(type='data', symmetric=False,
        array=combined_df['LOS 95% CI Upper'] - combined_df['Mean LOS'],
        arrayminus=combined_df['Mean LOS'] - combined_df['LOS 95% CI Lower'], visible=True)
))

fig_headline.add_trace(go.Scatter(
    x=combined_df['Scenario'],
    y=combined_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers+text',
    text=combined_df['Mean Dogs Rehomed'].astype(int),
    textposition='top center',
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_headline.update_layout(
    title='RQ3 Scenario 3 — Combined Interventions vs Individual Scenarios (Calibrated Model)',
    xaxis_title='Intervention',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(title='Dogs Rehomed per Year', overlaying='y', side='right'),
    height=550, width=1000,
    legend=dict(x=0.35, y=1.15)
)

fig_headline.write_image('../outputs/rq3_headline_comparison_calibrated.png', width=1200, height=650, scale=2)
fig_headline.write_html('../outputs/rq3_headline_comparison_calibrated.html')

fig_headline.show()

In [ ]:
# RQ3 Scenario 4 — Direct-Rehoming Support
# Strengthening direct rehoming = boosting P(Intake to Adoption)
# so more dogs go straight home without entering the Movement loop
# Baseline P_INTAKE_TO_ADOPTION = 0.420

def dog_journey_direct(env, dog_id, results, p_intake_adopt, p_intake_transfer):
    arrival_time = env.now
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < p_intake_adopt:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los})
        return
    elif rand < p_intake_adopt + p_intake_transfer:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1
        rand = np.random.random()
        total = P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER
        if rand < P_MOVEMENT_TO_ADOPTION / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los})

def arrival_gen_direct(env, results, p_intake_adopt, p_intake_transfer):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_direct(env, dog_id, results, p_intake_adopt, p_intake_transfer))

direct_scenarios = {
    'Baseline (42% direct)': 0.420,
    'Improved (50% direct)': 0.500,
    'Strong (58% direct)': 0.580,
    'High (66% direct)': 0.660,
}

N_REPLICATIONS = 100
direct_results = []

for scenario_name, p_intake_adopt in direct_scenarios.items():
    # Keep intake to transfer fixed; the rest goes to movement
    p_intake_transfer = P_INTAKE_TO_TRANSFER

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_direct(env, rep_results, p_intake_adopt, p_intake_transfer))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    direct_results.append({
        'Scenario': scenario_name,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
    })
    print(f"{scenario_name} done — Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

direct_df = pd.DataFrame(direct_results)
print("\nRQ3 Scenario 4 — Direct-Rehoming Support (Calibrated):")
print(direct_df.to_string(index=False))

In [ ]:
fig_direct = go.Figure()

fig_direct.add_trace(go.Bar(
    x=direct_df['Scenario'],
    y=direct_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(type='data', symmetric=False,
        array=direct_df['LOS 95% CI Upper'] - direct_df['Mean LOS'],
        arrayminus=direct_df['Mean LOS'] - direct_df['LOS 95% CI Lower'], visible=True)
))

fig_direct.add_trace(go.Scatter(
    x=direct_df['Scenario'],
    y=direct_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers+text',
    text=direct_df['Mean Dogs Rehomed'].astype(int),
    textposition='top center',
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_direct.update_layout(
    title='RQ3 Scenario 4 — Direct-Rehoming Support: LOS and Rehoming Throughput (Calibrated Model)',
    xaxis_title='Direct-Rehoming Level',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(title='Dogs Rehomed per Year', overlaying='y', side='right'),
    height=500, width=950,
    legend=dict(x=0.6, y=1.15)
)

fig_direct.write_image('../outputs/rq3_scenario4_direct_rehoming.png', width=1100, height=600, scale=2)
fig_direct.write_html('../outputs/rq3_scenario4_direct_rehoming.html')
fig_direct.show()

In [ ]:
# One-sample t-tests vs calibrated baseline (60.12 days)
baseline_mean = 60.12

scenarios_to_test = {
    'Foster Only (80%)': (56.67, (57.96 - 56.67) / 1.96),
    'Priority Only (high boost)': (56.88, (58.04 - 56.88) / 1.96),
    'Combined (Foster + Priority)': (54.62, (55.85 - 54.62) / 1.96),
}

print("Significance Tests vs Calibrated Baseline (60.12 days):\n")

significance_rows = []
for name, (mean, std) in scenarios_to_test.items():
    n = 100
    t_stat = (mean - baseline_mean) / (std / np.sqrt(n))
    p_value = scipy_stats.t.sf(abs(t_stat), df=n-1) * 2
    sig = 'Yes' if p_value < 0.05 else 'No'
    significance_rows.append({
        'Scenario': name,
        'Mean LOS': mean,
        'LOS Reduction': round(baseline_mean - mean, 2),
        't-statistic': round(t_stat, 3),
        'p-value': '<0.001' if p_value < 0.001 else round(p_value, 4),
        'Significant': sig
    })
    print(f"{name}: t={t_stat:.3f}, p={'<0.001' if p_value < 0.001 else f'{p_value:.4f}'}, Significant={sig}")

significance_df = pd.DataFrame(significance_rows)
print("\nRQ3 Significance Test Summary (Calibrated Model):")
print(significance_df.to_string(index=False))

In [ ]:
baseline_mean = 58.87  # calibrated baseline

direct_to_test = {
    'Improved (50% direct)': (55.48, (56.79 - 55.48) / 1.96),
    'Strong (58% direct)': (52.47, (53.66 - 52.47) / 1.96),
    'High (66% direct)': (49.13, (50.33 - 49.13) / 1.96),
}

print("Significance Tests — Direct-Rehoming vs Calibrated Baseline (58.87 days):\n")

for name, (mean, std) in direct_to_test.items():
    n = 100
    t_stat = (mean - baseline_mean) / (std / np.sqrt(n))
    p_value = scipy_stats.t.sf(abs(t_stat), df=n-1) * 2
    sig = 'Significant' if p_value < 0.05 else 'Not significant'
    print(f"{name}: t={t_stat:.3f}, p={'<0.001' if p_value < 0.001 else f'{p_value:.4f}'} — {sig}")

In [ ]:
# Recreate los_df for validation
intake_dates = affiliation_history[affiliation_history['Process'] == 'Intake']\
    .sort_values('Start Date')\
    .groupby('Dog ID')['Start Date'].first().reset_index()
intake_dates.columns = ['Dog ID', 'Intake Date']

last_records = affiliation_history.sort_values('Start Date')\
    .groupby('Dog ID').last().reset_index()
last_records['Exit Date'] = last_records['End Date'].fillna(last_records['Start Date'])

los_df = intake_dates.merge(last_records[['Dog ID', 'Exit Date']], on='Dog ID')
los_df['Length of Stay'] = (
    pd.to_datetime(los_df['Exit Date']) - pd.to_datetime(los_df['Intake Date'])
).dt.days

# QQ Plot — Real vs Calibrated Simulated LOS
real_los = los_df['Length of Stay'].dropna().values
real_los = real_los[real_los >= 0]

sim_los = mc_df['los'].sample(n=len(real_los), random_state=42).values

real_sorted = np.sort(real_los)
sim_sorted = np.sort(sim_los)

min_len = min(len(real_sorted), len(sim_sorted))
real_sorted = real_sorted[:min_len]
sim_sorted = sim_sorted[:min_len]

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(real_sorted, sim_sorted, alpha=0.3, s=5, color='steelblue', label='Quantile pairs')
ax.plot([0, max(real_sorted)], [0, max(real_sorted)], 'r--', lw=2, label='Perfect fit (y=x)')

ax.axvspan(0, 60, alpha=0.08, color='green', label='Good fit region (0–60 days, Normal)')

ax.annotate('Model closely follows\nreal data in this range\n(0–60 days, Normal)',
            xy=(30, 50), xytext=(150, 30),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=9, color='green')

ax.annotate('Simulation underestimates\nmedium-stay dogs (60–182 days)',
            xy=(120, 100), xytext=(300, 100),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

ax.annotate('Model rarely produces\nthe very longest stays\n(183+ days)',
            xy=(500, 250), xytext=(400, 320),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

ax.set_xlabel('Real LOS (days)', fontsize=12)
ax.set_ylabel('Simulated LOS (days)', fontsize=12)
ax.set_title('QQ Plot — Real vs Simulated LOS Distribution\n(Calibrated Model P=0.30, 100 Monte Carlo Replications)', fontsize=12)
ax.legend(fontsize=9)

ax.text(0.01, 0.99,
    'Points on the red line = perfect match\nPoints below = simulation underestimates LOS\nPoints above = simulation overestimates LOS',
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../outputs/qq_plot_validation_calibrated.png', dpi=150, bbox_inches='tight')
plt.show()